# 05_tuning_3. 그룹별 FICR 손실 + T_soft 재튜닝 (독립 실험)

**왜 별도 파일인가**: `src/nn.py`의 `soft_metric_loss`/`true_score`를 대회 공식 지표와 같은 **그룹별 평균 구조**로 고쳤다(이전엔 세 그룹을 한 덩어리로 합치는 pooled 방식이라, 채점 표본이 적은 group_3가 과소가중됐다 — HANDOFF 2026-07-24 우선순위 1). 그런데 `05_tuning_2` 0-10을 그대로 재실행하니 블렌드 3-fold 평균이 **0.6306 → 0.6256(-0.0050)** 로 오히려 나빠졌다.

**핵심 의심**: MLP의 하이퍼파라미터(특히 `T_soft=0.003`)는 전부 **옛 pooled 손실에 맞춰** 14·18절에서 튜닝된 값이다. 손실 구조가 바뀌면 FICR항과 NMAE항의 그룹별 균형·스케일이 달라져 **최적 T_soft도 이동**했을 수 있다. 즉 -0.0050은 "그룹별 손실이 나쁘다"가 아니라 "새 손실에 안 맞는 옛 T_soft를 썼다"는 것일 수 있다 — 공정한 비교가 아니다.

**공정성 원칙(중요)**: 비교 기준 0.6306은 **T_soft·블렌드 가중치를 모두 최적화한** 결과다. 그러니 새 손실도 T_soft만 맞춰 비교하면 불공정하다 — 새 손실이 지더라도 "손실이 나빠서"가 아니라 "블렌드 가중치가 옛 MLP 기준에 묶여 있어서"일 수 있다. 그래서 이 노트북은 **각 T_soft마다 블렌드 가중치까지 다시 최적화**(14-8과 같은 그룹별 그리드, 재학습 불필요·캐시 재사용)한 값으로 0.6306과 겨룬다. (MLP 구조/dropout/lr은 18절에서 "기본값이 이미 최적(+0.0000)"으로 확인돼 옛값 유지, tau는 CatBoost 전용이라 손실과 무관 — 이 둘은 잔여 비대칭으로 남기고 필요 시 4절 이후 논의.)

**이 노트북이 하는 일**: 새 그룹별 손실에서 **T_soft 스윕 + 각 T_soft에서 블렌드 가중치 재최적화** → 재최적 블렌드가 0.6306을 넘으면 seed 재검증, 못 넘으면 되돌리기 논의. 스윕하면서 **group_1/2/3 점수를 따로** 찍어, group_3이 실제로 오르는지도 함께 본다.

**실행 전 필수**: 커널을 새로 시작해 수정된 `src/nn.py`(그룹별 손실)가 로드되게 한다. `T_soft=0.003` 행이 `05_tuning_2`의 재실행값(블렌드 0.6256)을 재현하면 두 노트북이 일관됨을 확인한 셈이다.

**의존성**: 본 노트북은 완전 독립 실행(`05_tuning_2` 불필요). `data/processed/train_features_v2.parquet`만 있으면 된다.

## 0. 셋업 + 헬퍼 재현

`05_tuning_2` 0-1~0-5의 헬퍼(9·11·14절 압축)를 이 파일에 그대로 옮겨 self-contained하게 만든다. 확정 상수(τ=0.70, `DEFAULT_PARAMS`, 그룹별 블렌드 가중치)는 재탐색하지 않고 상수로 둔다.

### 0-1. 셋업 + 데이터 로드

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import torch

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH
from src import nn as mlp_nn

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 100)

train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("python:", sys.executable)
print("train_features:", train_features.shape, "| 피처 개수:", len(FEATURE_COLS), "| torch:", torch.__version__)

python: d:\공모전\wind_forecast\venv\Scripts\python.exe
train_features: (26304, 54) | 피처 개수: 50 | torch: 2.13.0+cpu


### 0-2. 3-fold CV 정의 (05_tuning_2 0-2와 동일)

In [2]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(f"{fold['name']}: train {ft.shape}, valid {fv.shape}")

fold1: train (13104, 54), valid (4416, 54)
fold2: train (17520, 54), valid (4368, 54)
fold3: train (21888, 54), valid (4416, 54)


### 0-3. CatBoost 학습·예측·채점 헬퍼 (3·9·11절 압축)

In [3]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]

DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}
best_tau = 0.70  # 9절 확정


def to_long_ext(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out


def train_fold_model(fold_train, params, feature_cols=None, early_stop_frac=0.15, seed=SEED,
                     quantile_alpha=None, use_sample_weight=False):
    feature_cols = feature_cols if feature_cols is not None else FEATURE_COLS
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)
    weight = fit_rows["actual_kwh"].to_numpy(dtype=float) if use_sample_weight else None
    fit_kwargs = {"sample_weight": weight} if weight is not None else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False, **fit_kwargs,
    )
    return model


def predict_group(model, fold_valid, g, feature_cols=None):
    feature_cols = feature_cols if feature_cols is not None else FEATURE_COLS
    features = feature_cols + ["group_id"]
    valid_g = fold_valid.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    return model.predict(valid_g[features]) * CAPACITY_KWH[g]


def single_group_score(fold_valid, g, pred_kwh):
    """단일 그룹만 metric.py 로직대로 직접 채점 → 그 그룹의 Score(0.5*(1-nmae)+0.5*ficr) 반환."""
    actual = fold_valid[g].to_numpy(dtype=float)
    pred = np.asarray(pred_kwh, dtype=float)
    capacity = CAPACITY_KWH[g]
    valid = actual >= capacity * 0.10
    actual_v, pred_v = actual[valid], pred[valid]
    error_rate = np.abs(pred_v - actual_v) / capacity
    nmae = float(np.mean(error_rate))
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = float(np.sum(actual_v * unit_price) / np.sum(actual_v * 4.0))
    return 0.5 * (1 - nmae) + 0.5 * ficr

print("CatBoost 헬퍼 준비 완료")

CatBoost 헬퍼 준비 완료


### 0-4. MLP 인프라 (14절 압축) — 수정된 `src/nn.py`(그룹별 손실)를 그대로 사용

In [4]:
def build_mlp_features(df, feature_cols, mu=None, sd=None):
    group_onehot = pd.get_dummies(df["group_id"].astype(int), prefix="grp").reindex(
        columns=[f"grp_{i}" for i in range(3)], fill_value=0
    ).to_numpy(dtype=np.float32)
    num_x = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    if mu is None:
        mu, sd = mlp_nn.fit_standardizer(num_x)
    num_x = mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32)
    x = np.concatenate([num_x, group_onehot], axis=1)
    return x, mu, sd


def train_mlp_fold(fold_train, feature_cols, seed=SEED, T_soft=0.003,
                   hidden=(256, 256), dropout=0.15, early_stop_frac=0.15, verbose=False):
    long_df = to_long_ext(fold_train, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)

    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10
    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10

    # group_id는 build_mlp_features가 X 마지막 3열에 원-핫으로 붙였으므로 train_mlp가 자동 복원한다.
    model, best_val_score, best_epoch = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, verbose=verbose,
    )
    return model, mu, sd, best_epoch


def predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd):
    valid_g = fold_valid.copy()
    valid_g["group_id"] = GROUP_ID_MAP[g]
    x, _, _ = build_mlp_features(valid_g, feature_cols, mu=mu, sd=sd)
    pred_util = mlp_nn.predict_mlp(model, x)
    return pred_util * CAPACITY_KWH[g]

print("MLP 헬퍼 준비 완료 (nn.py = 그룹별 손실 버전)")

MLP 헬퍼 준비 완료 (nn.py = 그룹별 손실 버전)


### 0-5. 확정 상수 + 비교 기준

- `GROUP_BLEND_BEST`: 14-8 확정 그룹별 블렌드 가중치(CatBoost 대 MLP). g3=1.0이라 group_3는 MLP 100%.
- 비교 기준은 **옛 pooled 손실 + T_soft=0.003으로 얻은 `05_tuning_2` 0-10 커밋 출력**: 블렌드 3-fold 0.6306, seed 3개 평균 0.6295(14-9). 새 손실은 이제 nn.py에 있으므로 옛 손실 수치를 여기서 재계산하지는 못한다(커밋 기록값을 기준으로 둔다).

In [5]:
GROUP_BLEND_BEST = {"kpx_group_1": 0.4, "kpx_group_2": 0.5, "kpx_group_3": 1.0}  # 14-8 확정
BASELINE_BLEND_3FOLD = 0.6306    # 옛 pooled 손실 + T_soft=0.003 (05_tuning_2 0-10 커밋 출력, seed=42)
BASELINE_BLEND_SEEDMEAN = 0.6295 # 14-9 기록값(seed 3개 평균, 표준편차 0.0013)
OLD_T_SOFT = 0.003

# CatBoost는 손실 변경과 무관 → fold별 예측을 한 번만 캐시(T_soft 스윕 내내 재사용).
CAT_FOLD = {}
for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    cat_model = train_fold_model(ft, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True)
    CAT_FOLD[fold["name"]] = {"fv": fv, "cat": {g: predict_group(cat_model, fv, g) for g in TARGET_COLS}}
print("CatBoost fold 예측 캐시 완료:", list(CAT_FOLD.keys()))

CatBoost fold 예측 캐시 완료: ['fold1', 'fold2', 'fold3']


## 1. T_soft 스윕 + 블렌드 가중치 재최적화 (새 그룹별 손실)

각 T_soft마다 3-fold(seed=42) MLP를 학습하고, 세 가지를 채점한다:
- **MLP 단독** (전체 + group_1/2/3)
- **고정 블렌드** — 옛 가중치(g1=0.4/g2=0.5/g3=1.0) 그대로. `T_soft=0.003` 행이 `05_tuning_2` 재실행(0.6256)을 재현하는지 내부 검산용.
- **재최적 블렌드** — 이 T_soft의 MLP에 맞춰 그룹별 블렌드 가중치를 0~1 그리드로 다시 최적화(14-8 방식, 재학습 없이 캐시 예측만 사용). **이게 0.6306과의 공정한 비교 대상.**

전체 점수는 3-fold `metric()` 평균, 그룹별은 각 fold에서 그 그룹만 채점한 Score의 3-fold 평균이다. Score는 그룹 점수의 단순평균이라 그룹별 가중치를 독립적으로 최적화해도 전체가 최대가 된다(14·19-7과 같은 논리).

In [6]:
def scores_3fold(pred_by_fold):
    """pred_by_fold[fold_name][g] = kwh 예측 → (전체 3-fold 평균, {g: 그룹별 3-fold 평균})."""
    overall, per_g = [], {g: [] for g in TARGET_COLS}
    for fold in FOLDS:
        fv = CAT_FOLD[fold["name"]]["fv"]
        preds = pred_by_fold[fold["name"]]
        s, _, _ = metric(make_answer_df(fv), make_pred_df(fv, preds))
        overall.append(s)
        for g in TARGET_COLS:
            per_g[g].append(single_group_score(fv, g, preds[g]))
    return float(np.mean(overall)), {g: float(np.mean(per_g[g])) for g in TARGET_COLS}


BLEND_GRID = [round(0.1 * i, 1) for i in range(11)]  # 0.0, 0.1, ..., 1.0


def optimize_blend_weights(mlp_by_fold, grid=BLEND_GRID):
    """이 MLP에 맞춰 그룹별 블렌드 가중치 w(=MLP 비중)를 다시 최적화.
    pred = (1-w)*CatBoost + w*MLP. 그룹마다 독립적으로 3-fold 평균 Score가 최대인 w를 고른다."""
    best_w, best_score = {}, {}
    for g in TARGET_COLS:
        top = (-1.0, None)
        for w in grid:
            fs = []
            for fold in FOLDS:
                fv = CAT_FOLD[fold["name"]]["fv"]
                pred = (1 - w) * CAT_FOLD[fold["name"]]["cat"][g] + w * mlp_by_fold[fold["name"]][g]
                fs.append(single_group_score(fv, g, pred))
            s = float(np.mean(fs))
            if s > top[0]:
                top = (s, w)
        best_score[g], best_w[g] = top
    overall = float(np.mean([best_score[g] for g in TARGET_COLS]))  # 전체 = 그룹별 최적 Score 평균
    return overall, best_w, best_score


T_GRID = [0.001, 0.002, 0.003, 0.004, 0.006, 0.010]
sweep_rows = []
mlp_cache = {}    # T -> {fold_name: {g: pred}}
reopt_cache = {}  # T -> best_w dict

for T in T_GRID:
    mlp_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=SEED, T_soft=T)
        mlp_by_fold[fold["name"]] = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
    mlp_cache[T] = mlp_by_fold

    mlp_overall, mlp_g = scores_3fold(mlp_by_fold)

    fixed_blend = {
        fold["name"]: {
            g: (1 - GROUP_BLEND_BEST[g]) * CAT_FOLD[fold["name"]]["cat"][g] + GROUP_BLEND_BEST[g] * mlp_by_fold[fold["name"]][g]
            for g in TARGET_COLS
        }
        for fold in FOLDS
    }
    fix_overall, _ = scores_3fold(fixed_blend)

    reopt_overall, best_w, reopt_g = optimize_blend_weights(mlp_by_fold)
    reopt_cache[T] = best_w

    sweep_rows.append({
        "T_soft": T,
        "MLP전체": mlp_overall, "MLP_g3": mlp_g["kpx_group_3"],
        "고정블렌드": fix_overall,
        "재최적블렌드": reopt_overall,
        "재최적_g1": reopt_g["kpx_group_1"], "재최적_g2": reopt_g["kpx_group_2"], "재최적_g3": reopt_g["kpx_group_3"],
        "w(g1/g2/g3)": f"{best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}",
    })
    print(f"T_soft={T:.3f} | MLP전체={mlp_overall:.4f}(g3={mlp_g['kpx_group_3']:.4f}) "
          f"| 고정블렌드={fix_overall:.4f} | 재최적블렌드={reopt_overall:.4f} "
          f"[w={best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}, "
          f"g3={reopt_g['kpx_group_3']:.4f}]")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

T_soft=0.001 | MLP전체=0.6259(g3=0.5934) | 고정블렌드=0.6285 | 재최적블렌드=0.6294 [w=0.6/0.3/1.0, g3=0.5934]
T_soft=0.002 | MLP전체=0.6279(g3=0.5950) | 고정블렌드=0.6306 | 재최적블렌드=0.6313 [w=0.3/0.3/1.0, g3=0.5950]
T_soft=0.003 | MLP전체=0.6205(g3=0.5937) | 고정블렌드=0.6290 | 재최적블렌드=0.6296 [w=0.2/0.4/0.8, g3=0.5938]
T_soft=0.004 | MLP전체=0.6223(g3=0.5910) | 고정블렌드=0.6279 | 재최적블렌드=0.6283 [w=0.3/0.6/1.0, g3=0.5910]
T_soft=0.006 | MLP전체=0.6220(g3=0.5932) | 고정블렌드=0.6288 | 재최적블렌드=0.6292 [w=0.3/0.5/1.0, g3=0.5932]
T_soft=0.010 | MLP전체=0.6222(g3=0.5950) | 고정블렌드=0.6295 | 재최적블렌드=0.6300 [w=0.3/0.5/0.8, g3=0.5951]


,T_soft,MLP전체,MLP_g3,고정블렌드,재최적블렌드,재최적_g1,재최적_g2,재최적_g3,w(g1/g2/g3)
0,0.001,0.625891,0.593428,0.628519,0.629423,0.647231,0.647609,0.593428,0.6/0.3/1.0
1,0.002,0.627892,0.595025,0.630649,0.631293,0.648512,0.650342,0.595025,0.3/0.3/1.0
2,0.003,0.620461,0.593683,0.629014,0.629625,0.644928,0.650176,0.593773,0.2/0.4/0.8
3,0.004,0.622259,0.590980,0.627939,0.628322,0.644650,0.649335,0.590980,0.3/0.6/1.0
4,0.006,0.621969,0.593178,0.628845,0.629175,0.645572,0.648775,0.593178,0.3/0.5/1.0
5,0.010,0.622238,0.595025,0.629520,0.630038,0.643908,0.651149,0.595057,0.3/0.5/0.8


## 1b. T_soft 상단 확장 (봉우리 감싸기)

1절에서 **재최적 블렌드가 T_soft를 키울수록 계속 올라 그리드 맨 끝(0.010)에서 최고(0.6300)** 였다 — 즉 새 손실의 진짜 최적 T_soft는 아직 안 잡혔다(경계에서 멈춤). 옛 설정(0.6306)과 겨우 −0.0006 차이라, 위쪽을 조금만 더 보면 넘을 수도 있다. `T_soft ∈ {0.015, 0.02, 0.03, 0.05}`를 추가로 학습해 **봉우리를 안쪽에 가두고**(최적이 경계가 아니게) 최종 판정한다. 최적이 여전히 경계면 그리드를 더 늘린다.

*(주의: T_soft가 커질수록 FICR 계단이 매우 완만해져 어느 지점부터는 실제 지표와 멀어지며 개선이 꺾인다 — 봉우리가 반드시 존재한다. 무한정 키우는 게 목적이 아니라 그 봉우리를 찾는 것.)*

In [7]:
# 1b. T_soft 상단 확장 — 1절 재최적블렌드가 그리드 끝(0.010)에서 아직 상승 중이라 위쪽을 더 본다.
# 커널에 이미 있는 헬퍼/캐시(scores_3fold, optimize_blend_weights, CAT_FOLD, sweep_rows, reopt_cache)를 재사용한다.
EXT_T_GRID = [0.015, 0.02, 0.025, 0.03, 0.035, 0.04, 0.045, 0.05]

for T in EXT_T_GRID:
    if any(abs(r["T_soft"] - T) < 1e-9 for r in sweep_rows):
        continue
    mlp_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=SEED, T_soft=T)
        mlp_by_fold[fold["name"]] = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
    mlp_cache[T] = mlp_by_fold

    mlp_overall, mlp_g = scores_3fold(mlp_by_fold)
    fixed_blend = {
        fold["name"]: {
            g: (1 - GROUP_BLEND_BEST[g]) * CAT_FOLD[fold["name"]]["cat"][g] + GROUP_BLEND_BEST[g] * mlp_by_fold[fold["name"]][g]
            for g in TARGET_COLS
        }
        for fold in FOLDS
    }
    fix_overall, _ = scores_3fold(fixed_blend)
    reopt_overall, best_w, reopt_g = optimize_blend_weights(mlp_by_fold)
    reopt_cache[T] = best_w

    sweep_rows.append({
        "T_soft": T,
        "MLP전체": mlp_overall, "MLP_g3": mlp_g["kpx_group_3"],
        "고정블렌드": fix_overall,
        "재최적블렌드": reopt_overall,
        "재최적_g1": reopt_g["kpx_group_1"], "재최적_g2": reopt_g["kpx_group_2"], "재최적_g3": reopt_g["kpx_group_3"],
        "w(g1/g2/g3)": f"{best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}",
    })
    print(f"T_soft={T:.3f} | MLP전체={mlp_overall:.4f}(g3={mlp_g['kpx_group_3']:.4f}) "
          f"| 재최적블렌드={reopt_overall:.4f} [w={best_w['kpx_group_1']:.1f}/{best_w['kpx_group_2']:.1f}/{best_w['kpx_group_3']:.1f}, "
          f"g3={reopt_g['kpx_group_3']:.4f}]")

sweep_df = pd.DataFrame(sweep_rows).sort_values("T_soft").reset_index(drop=True)
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
gap = best["재최적블렌드"] - BASELINE_BLEND_3FOLD
at_edge = abs(float(best["T_soft"]) - max(r["T_soft"] for r in sweep_rows)) < 1e-9

print(f"\n[전체 그리드 최적] T_soft={best['T_soft']:.3f}, 재최적블렌드={best['재최적블렌드']:.4f}, "
      f"개선폭={gap:+.4f} ({gap/0.0013:+.1f}×std), w={best['w(g1/g2/g3)']}")
if gap > 0 and not at_edge:
    print("→ 위쪽 확장으로 옛 설정을 넘었고 최적도 그리드 내부. seed 재검증 가치 있음(아래 3절 로직 재실행).")
elif at_edge:
    print("→ 최적이 여전히 그리드 맨 끝. EXT_T_GRID를 더 위로 늘려 봉우리를 감싸야 함(아직 T_soft 레버 안 소진).")
else:
    print("→ 위로 확장해도 옛 설정(0.6306)을 못 넘음. T_soft 레버는 여기서 소진 — 다음은 구조/lr 재튜닝 여부 결정.")
sweep_df

T_soft=0.015 | MLP전체=0.6220(g3=0.5943) | 재최적블렌드=0.6299 [w=0.2/0.5/0.8, g3=0.5943]
T_soft=0.020 | MLP전체=0.6223(g3=0.5935) | 재최적블렌드=0.6308 [w=0.2/0.6/0.8, g3=0.5947]
T_soft=0.025 | MLP전체=0.6241(g3=0.5949) | 재최적블렌드=0.6317 [w=0.2/0.7/0.8, g3=0.5951]
T_soft=0.030 | MLP전체=0.6246(g3=0.5959) | 재최적블렌드=0.6323 [w=0.3/0.7/0.9, g3=0.5963]
T_soft=0.035 | MLP전체=0.6241(g3=0.5951) | 재최적블렌드=0.6323 [w=0.3/0.6/0.8, g3=0.5957]
T_soft=0.040 | MLP전체=0.6241(g3=0.5944) | 재최적블렌드=0.6322 [w=0.3/0.7/1.0, g3=0.5944]
T_soft=0.045 | MLP전체=0.6237(g3=0.5941) | 재최적블렌드=0.6321 [w=0.2/0.7/0.8, g3=0.5941]
T_soft=0.050 | MLP전체=0.6227(g3=0.5921) | 재최적블렌드=0.6320 [w=0.2/0.7/0.8, g3=0.5941]

[전체 그리드 최적] T_soft=0.035, 재최적블렌드=0.6323, 개선폭=+0.0017 (+1.3×std), w=0.3/0.6/0.8
→ 위쪽 확장으로 옛 설정을 넘었고 최적도 그리드 내부. seed 재검증 가치 있음(아래 3절 로직 재실행).


,T_soft,MLP전체,MLP_g3,고정블렌드,재최적블렌드,재최적_g1,재최적_g2,재최적_g3,w(g1/g2/g3)
0,0.001,0.625891,0.593428,0.628519,0.629423,0.647231,0.647609,0.593428,0.6/0.3/1.0
1,0.002,0.627892,0.595025,0.630649,0.631293,0.648512,0.650342,0.595025,0.3/0.3/1.0
2,0.003,0.620461,0.593683,0.629014,0.629625,0.644928,0.650176,0.593773,0.2/0.4/0.8
3,0.004,0.622259,0.590980,0.627939,0.628322,0.644650,0.649335,0.590980,0.3/0.6/1.0
4,0.006,0.621969,0.593178,0.628845,0.629175,0.645572,0.648775,0.593178,0.3/0.5/1.0
5,0.010,0.622238,0.595025,0.629520,0.630038,0.643908,0.651149,0.595057,0.3/0.5/0.8
6,0.015,0.621981,0.594287,0.629445,0.629938,0.643797,0.651702,0.594314,0.2/0.5/0.8
7,0.020,0.622283,0.593465,0.629359,0.630790,0.645792,0.651878,0.594701,0.2/0.6/0.8
8,0.025,0.624133,0.594871,0.630990,0.631715,0.645188,0.654848,0.595107,0.2/0.7/0.8
9,0.030,0.624603,0.595867,0.631209,0.632287,0.645649,0.654920,0.596292,0.3/0.7/0.9


## 2. 비교 판정

새 손실의 **재최적 블렌드 최고값**(T_soft·블렌드 가중치 둘 다 새 손실에 맞춘 것)을 옛 설정(0.6306, 옛 손실에서 T_soft·블렌드 최적화한 것)과 비교한다 — 이제 양쪽 다 각자의 손실에서 두 knob을 최적화했으니 공정하다. 표준편차는 아직 단일 seed라 없지만, 14-9의 블렌드 seed 표준편차 0.0013을 참고 눈금으로 쓴다(개선폭이 그 수배는 돼야 seed 재검증 가치가 있다).

In [8]:
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
best_T = float(best["T_soft"])
best_w = reopt_cache[best_T]
gap = best["재최적블렌드"] - BASELINE_BLEND_3FOLD

print(f"새 손실 최적: T_soft={best_T:.3f}, 재최적 블렌드 3-fold={best['재최적블렌드']:.4f}  (가중치 {best['w(g1/g2/g3)']})")
print(f"옛 설정(pooled 손실 + T_soft=0.003 + 블렌드 0.4/0.5/1.0): 블렌드 3-fold={BASELINE_BLEND_3FOLD:.4f}")
print(f"개선폭: {gap:+.4f}  (참고: 블렌드 seed 표준편차 0.0013의 {gap/0.0013:+.1f}배)")

# 내부 검산: T_soft=0.003 고정블렌드가 05_tuning_2 재실행(0.6256)을 재현하는지
chk = sweep_df.loc[sweep_df['T_soft'] == 0.003, '고정블렌드']
if len(chk):
    print(f"\n[검산] T_soft=0.003 고정블렌드={chk.iloc[0]:.4f} (05_tuning_2 재실행 0.6256과 대조)")
print("[group_3 재최적 블렌드 점수 T_soft별]", {round(r['T_soft'],3): round(r['재최적_g3'],4) for r in sweep_rows})

if gap > 0:
    print("\n→ 새 손실이 (T_soft+블렌드 재최적 후) 옛 설정을 넘음. 아래 3절 seed 재검증 진행.")
else:
    print("\n→ T_soft·블렌드를 새 손실에 맞춰 재최적화해도 옛 설정을 못 넘음. 남은 잔여 비대칭(구조/lr)이 크지 않다면(18절: 기본값이 이미 최적) 그룹별 손실은 채택하지 않고 되돌리기 후보 — 4절에서 최종 논의.")

새 손실 최적: T_soft=0.035, 재최적 블렌드 3-fold=0.6323  (가중치 0.3/0.6/0.8)
옛 설정(pooled 손실 + T_soft=0.003 + 블렌드 0.4/0.5/1.0): 블렌드 3-fold=0.6306
개선폭: +0.0017  (참고: 블렌드 seed 표준편차 0.0013의 +1.3배)

[검산] T_soft=0.003 고정블렌드=0.6290 (05_tuning_2 재실행 0.6256과 대조)
[group_3 재최적 블렌드 점수 T_soft별] {0.001: 0.5934, 0.002: 0.595, 0.003: 0.5938, 0.004: 0.591, 0.006: 0.5932, 0.01: 0.5951, 0.015: 0.5943, 0.02: 0.5947, 0.025: 0.5951, 0.03: 0.5963, 0.035: 0.5957, 0.04: 0.5944, 0.045: 0.5941, 0.05: 0.5941}

→ 새 손실이 (T_soft+블렌드 재최적 후) 옛 설정을 넘음. 아래 3절 seed 재검증 진행.


## 3. seed 재검증 (조건부)

2절에서 재최적 블렌드가 0.6306을 넘은 경우에만 seed 3개(42/7/2024)로 재검증한다. 최적 T_soft와 **그 T_soft에서 재최적화된 블렌드 가중치**를 그대로 써서, `blend_seed_mean=0.6295`(14-9)와 비교해 개선폭이 표준편차 대비 충분히 큰지(이 프로젝트 채택 기준: 여러 배) 확인한다. 못 넘었으면 이 셀은 건너뛴다.

In [9]:
best = sweep_df.sort_values("재최적블렌드", ascending=False).iloc[0]
best_T = float(best["T_soft"])
best_w = reopt_cache[best_T]

if best["재최적블렌드"] <= BASELINE_BLEND_3FOLD:
    print("2절에서 옛 설정을 못 넘어 seed 재검증 생략 — 되돌리기 후보.")
else:
    print(f"seed 재검증 대상: T_soft={best_T:.3f}, 블렌드 가중치 {best['w(g1/g2/g3)']}")
    seed_scores = []
    for seed in [42, 7, 2024]:
        blend_by_fold = {}
        for fold in FOLDS:
            ft, _ = make_fold_frames(fold)
            fv = CAT_FOLD[fold["name"]]["fv"]
            model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=seed, T_soft=best_T)
            mlp_pred = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
            blend_by_fold[fold["name"]] = {
                g: (1 - best_w[g]) * CAT_FOLD[fold["name"]]["cat"][g] + best_w[g] * mlp_pred[g]
                for g in TARGET_COLS
            }
        ov, _ = scores_3fold(blend_by_fold)
        seed_scores.append(ov)
        print(f"seed={seed}: 재최적 블렌드 3-fold={ov:.4f}")

    m, sd_ = float(np.mean(seed_scores)), float(np.std(seed_scores))
    print(f"\nseed 3개 평균={m:.4f}, 표준편차={sd_:.4f}")
    gap = m - BASELINE_BLEND_SEEDMEAN
    print(f"옛 블렌드 seed 평균(0.6295) 대비 {gap:+.4f} (표준편차의 {gap/sd_ if sd_>0 else float('nan'):+.1f}배)")

seed 재검증 대상: T_soft=0.035, 블렌드 가중치 0.3/0.6/0.8
seed=42: 재최적 블렌드 3-fold=0.6323
seed=7: 재최적 블렌드 3-fold=0.6326
seed=2024: 재최적 블렌드 3-fold=0.6299

seed 3개 평균=0.6316, 표준편차=0.0012
옛 블렌드 seed 평균(0.6295) 대비 +0.0021 (표준편차의 +1.7배)


## 4. 종합 해석

### 4-1. 결과 요약 (2026-07-24 집에서 완주)

| 단계 | 최적 T_soft | 재최적 블렌드 3-fold | 옛 설정 대비 |
|---|---|---|---|
| 1절 스윕 [0.001~0.010] | 0.002 | 0.6313 | +0.0007 (경계에서 멈춤) |
| **1b절 상단 확장 [0.015~0.050]** | **0.030~0.035** | **0.6323** | **+0.0017 (1.3×std)** |
| **3절 seed 재검증 (42/7/2024)** | 0.035 | **평균 0.6316 (표준편차 0.0012)** | **+0.0021 (1.7×std)** |

- **봉우리 확정**: 재최적 블렌드가 T_soft=0.030~0.035에서 정점(0.6323)을 찍고 양옆으로 완만히 하강(0.050=0.6320) — 최적이 그리드 내부에 확실히 갇혔다. T_soft 레버는 여기서 소진.
- **최종 채택값**: T_soft=0.035, 블렌드 가중치 group_1=0.3 / group_2=0.6 / group_3=0.8.

### 4-2. 우선순위 1 가설이 방향으로 검증됨 (group_3이 실제로 올라감)

이 실험의 출발점은 "pooled 손실이 채점 표본 적은 group_3을 과소가중한다 → 그룹별 평균 구조로 고치면 최약체 group_3이 개선될 것"(HANDOFF 우선순위 1)이었다. `재최적_g3` 열을 보면 그 방향이 그대로 나타난다: T_soft=0.001에서 0.5934 → **T_soft=0.030에서 0.5963(정점)**. 새 그룹별 손실 + 그에 맞는 T_soft가 실제로 최약체 그룹을 끌어올렸다. 부수적으로 **group_3 블렌드 가중치가 1.0 → 0.8로 내려갔다** — 옛 pooled 손실에선 "group_3은 MLP 100%가 최적"이었는데, 손실을 고치니 더는 그렇지 않다. 손실 수정이 모델 구조(블렌드 비중)를 실제로 바꿨다는 직접 증거다.

### 4-3. 판정 — 애매한 크기(1.7배)지만 성격이 다른 신호

이 프로젝트의 채택 기준은 "seed 표준편차의 여러 배"(9절 47.6배, 14절 6.2배 채택)다. +0.0021/1.7배는 그 기준에 못 미치고, **크기만 보면 19절 XGBoost(1.4배, 리더보드에서 -0.0024 역행)와 위험 구간이 겹친다.** 그러나 성격이 근본적으로 다르다:

- **XGBoost**: 블렌드에 새 모델을 하나 더 얹는 *자유도 추가* → CV에만 유리하고 LB에서 뒤집힐 유형(실제로 역행).
- **이번**: `soft_metric_loss`를 대회 지표(그룹별 평균)와 *구조적으로 일치*시킨 **버그성 수정** → 새 knob이 아니라 잘못 정렬돼 있던 걸 고친 것. CV 과적합 위험이 낮고, 지표에 더 맞게 고친 것이라 LB에서도 유지될 개연성이 상대적으로 높다.

이 성격 차이 때문에 CV 크기만으로 기각하지 않고 **리더보드로 직접 확인**하기로 결정(민석님, 2026-07-24).

### 4-4. 한계 — 검산 불일치(측정 환경 차이)

정직하게 남긴다: 2절 검산에서 `T_soft=0.003 고정블렌드=0.6290`이 나왔는데, HANDOFF의 학교 실행 기록은 **0.6256**이었다(0.0034 차이). 같은 노트북·같은 nn.py인데 값이 다른 건 **측정 환경(집↔학교 머신)에서 MLP 학습이 완벽히 재현되지 않기 때문**으로 보인다(torch/BLAS 부동소수점 차이). 이 때문에 비교 기준 `0.6306/0.6295`(옛 pooled, 05_tuning_2에서 하드코딩)와 새 값 `0.6316`이 **완전히 동일한 환경에서 나란히 잰 비교는 아니다** — +0.0021 안에 "손실 개선"뿐 아니라 환경 차이가 섞였을 여지가 있다. 이 오염은 CV로는 걷어내기 어렵고, **결국 LB가 최종 심판**이라는 점이 이번에 CV 정밀도를 더 따지지 않고 제출로 넘어가는 또 하나의 이유다.

### 4-5. 다음 행동

- `src/nn.py`(그룹별 손실) 유지 + `train.ipynb`를 T_soft=0.035·블렌드 0.3/0.6/0.8로 갱신(완료) → 전체 재학습 → `inference.ipynb`로 `v4` 제출 파일 생성 → 민석님이 제출.
- **LB 결과로 최종 결정**: 2차 제출(0.625596)을 넘으면 채택 확정, 역행하면 `src/nn.py`를 pooled로 되돌리고 train/inference를 v2 구성(T_soft=0.003·블렌드 0.4/0.5/1.0)으로 복원.
- 관전 포인트: **CV +0.0021(1.7배)이라는 약한 신호가 LB에서 방향을 유지하는지** — XGBoost(1.4배)는 역행했지만 이번은 "지표에 손실을 맞춘 수정"이라 다른 결과를 기대한다. CV-LB 괴리(약한 신호는 실전에서 뒤집힐 수 있음)를 감안해 결과를 겸손하게 해석한다.

## 5. MLP 구조·정규화 재튜닝 — 새 그룹별 손실 위에서 (18절 방식 + fold3 안전장치)

**왜 다시 하나**: 18절에서 구조(256-256)·dropout(0.15)·lr(1e-3)을 재탐색했을 때 "기본값이 이미 최적(+0.0000)"이었지만, 그건 **옛 pooled 손실 기준**이었다. 이번에 손실을 그룹별 평균 구조로 바꾸고 T_soft가 0.003→0.035로 크게 이동했으니(1b절), 구조·정규화의 최적점도 옮겨졌을 수 있다. v4 채택(LB 0.629122)으로 이 손실이 실전에서 먹히는 게 확인됐으므로, 그 위에서 한 번 더 판다.

**안전장치(18절과 동일 — 외부 phase8 "CV는 좋아지고 홀드아웃은 역행" 교훈)**: ① 3-fold 평균과 함께 **fold3(가장 미래 방향)을 항상 따로** 본다, ② 최종 후보는 seed 3개(42/7/2024) 재검증을 거친 뒤에만 채택, ③ dropout은 과최적화 위험이 큰 0.40보다 낮게(0.05~0.35).

**순차 탐색**: 구조 → dropout → 학습률 → weight_decay(18절에서 미탐색으로 남겨둔 후보) 순으로 하나씩 고정해가며 좁힌다. 각 후보는 MLP 단독 3-fold·fold3와 함께 **재최적 블렌드**(0-4의 `optimize_blend_weights`, 캐시 예측만 써서 거의 공짜)까지 찍는다. 최종 판정 기준은 3절 seed 재검증값(재최적 블렌드 seed 평균 **0.6316**).

**의존성**: 0절 헬퍼(`CAT_FOLD`·MLP 인프라·`optimize_blend_weights`·`scores_3fold`)만 있으면 단독 실행 가능. `BEST_T_SOFT=0.035`를 상수로 가져오므로 1~4절 재실행 불필요.

**실행 시간**: 스윕마다 fold별 MLP를 새로 학습한다(구조 5 + dropout 5 + lr 4 + wd 3 ≈ 17조합 × 3fold ≈ 51회 + seed 재검증). 그리드는 각 셀 상단 상수로 노출했으니 오래 걸리면 줄여도 된다.

### 5-1. 헬퍼(lr/weight_decay 노출) + baseline 재확인

`nn.train_mlp`가 이미 `lr`/`weight_decay`를 인자로 받으므로, 0-4의 `train_mlp_fold`에 그 둘만 노출한 `train_mlp_fold_h`를 정의한다(안 주면 baseline과 동일 동작). `mlp_cv`는 한 설정으로 3-fold MLP를 학습해 (전체 Score, fold3 단독 Score)를 돌려준다. baseline(256-256/0.15/1e-3/wd 1e-4, T_soft=0.035)을 먼저 찍어 비교 기준을 만든다.

In [10]:
# 5절: 새 그룹별 손실(nn.py) 위에서 MLP 구조·정규화 재튜닝 (18절 방식 + fold3 안전장치)
BEST_T_SOFT = 0.035            # 1b절 확정(그룹별 손실 봉우리)
BASE_HIDDEN = (256, 256)
BASE_DROPOUT = 0.15
BASE_LR = 1e-3
BASE_WD = 1e-4
BASE_BLEND_SEEDMEAN = 0.6316   # 3절 seed 재검증(그룹별 손실+T_soft=0.035) 재최적 블렌드 seed 평균 = 최종 판정 기준


def train_mlp_fold_h(fold_train, feature_cols, seed=SEED, T_soft=BEST_T_SOFT,
                     hidden=BASE_HIDDEN, dropout=BASE_DROPOUT, lr=BASE_LR, weight_decay=BASE_WD,
                     early_stop_frac=0.15):
    # train_mlp_fold의 lr/weight_decay 노출판. build_mlp_features가 group_id 원-핫을 뒤에 붙여 train_mlp가 자동 복원한다.
    long_df = to_long_ext(fold_train, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]
    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)
    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10
    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10
    model, _, _ = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, lr=lr, weight_decay=weight_decay,
    )
    return model, mu, sd


def mlp_cv(seed=SEED, T_soft=BEST_T_SOFT, hidden=BASE_HIDDEN, dropout=BASE_DROPOUT, lr=BASE_LR, weight_decay=BASE_WD):
    # MLP 3-fold 학습 -> (mlp_by_fold, MLP 3-fold 전체 Score, MLP fold3 단독 Score)
    mlp_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd = train_mlp_fold_h(ft, FEATURE_COLS, seed=seed, T_soft=T_soft,
                                         hidden=hidden, dropout=dropout, lr=lr, weight_decay=weight_decay)
        mlp_by_fold[fold["name"]] = {g: predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd) for g in TARGET_COLS}
    overall, _ = scores_3fold(mlp_by_fold)
    fv3 = CAT_FOLD["fold3"]["fv"]
    s3, _, _ = metric(make_answer_df(fv3), make_pred_df(fv3, mlp_by_fold["fold3"]))
    return mlp_by_fold, overall, s3


def fmt_w(w):
    return f"{w['kpx_group_1']:.1f}/{w['kpx_group_2']:.1f}/{w['kpx_group_3']:.1f}"


base_mlp, base_overall, base_fold3 = mlp_cv()
base_reopt, base_w, _ = optimize_blend_weights(base_mlp)
print(f"[baseline] hidden{BASE_HIDDEN} drop{BASE_DROPOUT} lr{BASE_LR} wd{BASE_WD} (T_soft={BEST_T_SOFT})")
print(f"  MLP 3-fold={base_overall:.4f} | MLP fold3={base_fold3:.4f} | 재최적블렌드={base_reopt:.4f} [w={fmt_w(base_w)}]")
print(f"  최종 판정 기준(3절 seed 평균) = {BASE_BLEND_SEEDMEAN}")

[baseline] hidden(256, 256) drop0.15 lr0.001 wd0.0001 (T_soft=0.035)
  MLP 3-fold=0.6241 | MLP fold3=0.6536 | 재최적블렌드=0.6323 [w=0.3/0.6/0.8]
  최종 판정 기준(3절 seed 평균) = 0.6316


### 5-2. 은닉층 폭·깊이 스윕

dropout·lr·wd·T_soft를 baseline에 고정하고 구조만 바꾼다. **fold3을 별도 컬럼으로 남겨** 평균만 보고 판단하지 않는다(18절에서 평균 1위 `(256,256,256)`이 fold3에서는 역행했던 함정). fold3 기준으로 `best_hidden`을 고른다 — 표를 보고 아래 값을 직접 덮어써도 된다.

In [14]:
# 5-2. 은닉층 폭·깊이 스윕 (dropout=0.15, lr=1e-3, wd=1e-4, T_soft=0.035 고정)
HIDDEN_GRID = [(256, 256), (256, 128), (128, 128), (256, 256, 256), (256, 256, 256, 256), (256, 256, 256, 128), (384, 384)]

struct_rows = []
for h in HIDDEN_GRID:
    mlp_bf, ov, s3 = mlp_cv(hidden=h)
    reopt, w, _ = optimize_blend_weights(mlp_bf)
    struct_rows.append({"hidden": str(h), "MLP전체": ov, "MLP_fold3": s3, "재최적블렌드": reopt, "w(g1/g2/g3)": fmt_w(w)})
    print(f"hidden={str(h):16} | MLP전체={ov:.4f} | MLP_fold3={s3:.4f} | 재최적블렌드={reopt:.4f} [w={fmt_w(w)}]")

struct_df = pd.DataFrame(struct_rows)
best_hidden = HIDDEN_GRID[int(struct_df["MLP_fold3"].values.argmax())]  # fold3 우선(18절 원칙)
print(f"\n-> fold3 기준 best_hidden = {best_hidden} (baseline={BASE_HIDDEN})")
struct_df

hidden=(256, 256)       | MLP전체=0.6241 | MLP_fold3=0.6536 | 재최적블렌드=0.6323 [w=0.3/0.6/0.8]
hidden=(256, 128)       | MLP전체=0.6210 | MLP_fold3=0.6456 | 재최적블렌드=0.6304 [w=0.3/0.5/0.8]
hidden=(128, 128)       | MLP전체=0.6230 | MLP_fold3=0.6431 | 재최적블렌드=0.6304 [w=0.2/0.4/0.9]
hidden=(256, 256, 256)  | MLP전체=0.6279 | MLP_fold3=0.6515 | 재최적블렌드=0.6346 [w=0.4/0.5/1.0]
hidden=(256, 256, 256, 256) | MLP전체=0.6265 | MLP_fold3=0.6531 | 재최적블렌드=0.6337 [w=0.2/0.4/0.9]
hidden=(256, 256, 256, 128) | MLP전체=0.6274 | MLP_fold3=0.6510 | 재최적블렌드=0.6334 [w=0.2/0.4/0.9]
hidden=(384, 384)       | MLP전체=0.6249 | MLP_fold3=0.6529 | 재최적블렌드=0.6333 [w=0.3/0.5/0.9]

-> fold3 기준 best_hidden = (256, 256) (baseline=(256, 256))


,hidden,MLP전체,MLP_fold3,재최적블렌드,w(g1/g2/g3)
0,"(256, 256)",0.624090,0.653631,0.632346,0.3/0.6/0.8
1,"(256, 128)",0.620967,0.645580,0.630361,0.3/0.5/0.8
2,"(128, 128)",0.622972,0.643064,0.630430,0.2/0.4/0.9
3,"(256, 256, 256)",0.627947,0.651487,0.634607,0.4/0.5/1.0
4,"(256, 256, 256, 256)",0.626489,0.653143,0.633746,0.2/0.4/0.9
5,"(256, 256, 256, 128)",0.627434,0.650962,0.633429,0.2/0.4/0.9
6,"(384, 384)",0.624864,0.652869,0.633325,0.3/0.5/0.9


### 5-3. dropout 스윕

`best_hidden`을 고정하고 dropout만 바꾼다. 외부가 과최적화를 겪은 0.40보다 낮은 0.05~0.35 범위. fold3 기준으로 `best_dropout`을 고른다.

In [15]:
# 5-3. dropout 스윕 (best_hidden 고정)
DROPOUT_GRID = [0.05, 0.10, 0.15, 0.25, 0.35]

drop_rows = []
for d in DROPOUT_GRID:
    mlp_bf, ov, s3 = mlp_cv(hidden=best_hidden, dropout=d)
    reopt, w, _ = optimize_blend_weights(mlp_bf)
    drop_rows.append({"dropout": d, "MLP전체": ov, "MLP_fold3": s3, "재최적블렌드": reopt, "w(g1/g2/g3)": fmt_w(w)})
    print(f"dropout={d:.2f} | MLP전체={ov:.4f} | MLP_fold3={s3:.4f} | 재최적블렌드={reopt:.4f} [w={fmt_w(w)}]")

drop_df = pd.DataFrame(drop_rows)
best_dropout = float(DROPOUT_GRID[int(drop_df["MLP_fold3"].values.argmax())])
print(f"\n-> fold3 기준 best_dropout = {best_dropout} (baseline={BASE_DROPOUT})")
drop_df

dropout=0.05 | MLP전체=0.6263 | MLP_fold3=0.6515 | 재최적블렌드=0.6327 [w=0.3/0.7/0.9]
dropout=0.10 | MLP전체=0.6237 | MLP_fold3=0.6526 | 재최적블렌드=0.6323 [w=0.3/0.6/0.9]
dropout=0.15 | MLP전체=0.6241 | MLP_fold3=0.6536 | 재최적블렌드=0.6323 [w=0.3/0.6/0.8]
dropout=0.25 | MLP전체=0.6240 | MLP_fold3=0.6535 | 재최적블렌드=0.6318 [w=0.3/0.7/1.0]
dropout=0.35 | MLP전체=0.6234 | MLP_fold3=0.6557 | 재최적블렌드=0.6306 [w=0.3/0.6/0.7]

-> fold3 기준 best_dropout = 0.35 (baseline=0.15)


,dropout,MLP전체,MLP_fold3,재최적블렌드,w(g1/g2/g3)
0,0.05,0.626326,0.651517,0.632706,0.3/0.7/0.9
1,0.10,0.623697,0.652612,0.632289,0.3/0.6/0.9
2,0.15,0.624090,0.653631,0.632346,0.3/0.6/0.8
3,0.25,0.624003,0.653479,0.631750,0.3/0.7/1.0
4,0.35,0.623431,0.655747,0.630633,0.3/0.6/0.7


### 5-4. 학습률 → weight_decay 순차 스윕

`best_hidden`·`best_dropout`을 고정하고 학습률을 먼저 좁힌 뒤, 그 `best_lr`에서 weight_decay를 본다. **weight_decay는 18절에서 미탐색으로 남겨둔 후보**라 이번에 함께 확인한다.

In [16]:
# 5-4. 학습률 -> weight_decay 순차 스윕 (best_hidden, best_dropout 고정)
LR_GRID = [5e-4, 1e-3, 2e-3, 3e-3]
lr_rows = []
for lr in LR_GRID:
    mlp_bf, ov, s3 = mlp_cv(hidden=best_hidden, dropout=best_dropout, lr=lr)
    reopt, _, _ = optimize_blend_weights(mlp_bf)
    lr_rows.append({"lr": lr, "MLP전체": ov, "MLP_fold3": s3, "재최적블렌드": reopt})
    print(f"lr={lr:.4f} | MLP전체={ov:.4f} | MLP_fold3={s3:.4f} | 재최적블렌드={reopt:.4f}")
lr_df = pd.DataFrame(lr_rows)
best_lr = float(LR_GRID[int(lr_df["MLP_fold3"].values.argmax())])
print(f"-> fold3 기준 best_lr = {best_lr} (baseline={BASE_LR})\n")

WD_GRID = [0.0, 1e-4, 1e-3]
wd_rows = []
for wd in WD_GRID:
    mlp_bf, ov, s3 = mlp_cv(hidden=best_hidden, dropout=best_dropout, lr=best_lr, weight_decay=wd)
    reopt, _, _ = optimize_blend_weights(mlp_bf)
    wd_rows.append({"weight_decay": wd, "MLP전체": ov, "MLP_fold3": s3, "재최적블렌드": reopt})
    print(f"wd={wd:.4f} | MLP전체={ov:.4f} | MLP_fold3={s3:.4f} | 재최적블렌드={reopt:.4f}")
wd_df = pd.DataFrame(wd_rows)
best_wd = float(WD_GRID[int(wd_df["MLP_fold3"].values.argmax())])
print(f"-> fold3 기준 best_wd = {best_wd} (baseline={BASE_WD})")
wd_df

lr=0.0005 | MLP전체=0.6193 | MLP_fold3=0.6487 | 재최적블렌드=0.6282
lr=0.0010 | MLP전체=0.6234 | MLP_fold3=0.6557 | 재최적블렌드=0.6306
lr=0.0020 | MLP전체=0.6207 | MLP_fold3=0.6554 | 재최적블렌드=0.6288
lr=0.0030 | MLP전체=0.6258 | MLP_fold3=0.6574 | 재최적블렌드=0.6319
-> fold3 기준 best_lr = 0.003 (baseline=0.001)

wd=0.0000 | MLP전체=0.6231 | MLP_fold3=0.6573 | 재최적블렌드=0.6298
wd=0.0001 | MLP전체=0.6258 | MLP_fold3=0.6574 | 재최적블렌드=0.6319
wd=0.0010 | MLP전체=0.6224 | MLP_fold3=0.6552 | 재최적블렌드=0.6299
-> fold3 기준 best_wd = 0.0001 (baseline=0.0001)


,weight_decay,MLP전체,MLP_fold3,재최적블렌드
0,0.0000,0.623095,0.657312,0.629758
1,0.0001,0.625760,0.657370,0.631940
2,0.0010,0.622412,0.655237,0.629905


### 5-5. 최종 최적 조합 vs baseline → 조건부 seed 재검증

채택 조건(18절 안전장치): **(a) 재최적 블렌드(단일 seed)가 baseline을 넘고, (b) fold3이 baseline보다 나빠지지 않을 것.** 둘 다 만족할 때만 seed 3개(42/7/2024)로 재검증하고, 3절 기준(0.6316)과 표준편차 대비 개선폭을 본다. 순차 탐색 결과가 baseline과 같으면(18절처럼 "이미 최적") 재검증을 생략한다.

In [17]:
# 5-5. 최종 최적 조합 vs baseline -> 조건부 seed 재검증
final_cfg = dict(hidden=best_hidden, dropout=best_dropout, lr=best_lr, weight_decay=best_wd)
print(f"최적 조합: {final_cfg}")
print(f"baseline : hidden{BASE_HIDDEN} drop{BASE_DROPOUT} lr{BASE_LR} wd{BASE_WD}")

same_as_base = (tuple(best_hidden) == BASE_HIDDEN and best_dropout == BASE_DROPOUT
                and best_lr == BASE_LR and best_wd == BASE_WD)
if same_as_base:
    print("\n-> 순차 탐색 결과가 baseline과 완전히 동일. 새 손실에서도 기존 하이퍼파라미터가 최적 -> 변경 없음, seed 재검증 불필요.")
else:
    mlp42, ov42, s3_42 = mlp_cv(seed=42, **final_cfg)
    reopt42, final_w, _ = optimize_blend_weights(mlp42)
    print(f"\n[seed=42] 재최적블렌드={reopt42:.4f} (baseline {base_reopt:.4f}) | fold3={s3_42:.4f} (baseline {base_fold3:.4f})")
    if reopt42 <= base_reopt or s3_42 < base_fold3:
        print("-> baseline 미달 또는 fold3 역행 -> 채택하지 않음(seed 재검증 생략).")
    else:
        seed_scores = []
        for seed in [42, 7, 2024]:
            mlp_bf, _, _ = mlp_cv(seed=seed, **final_cfg)
            blend = {}
            for fold in FOLDS:
                fn = fold["name"]
                blend[fn] = {g: (1 - final_w[g]) * CAT_FOLD[fn]["cat"][g] + final_w[g] * mlp_bf[fn][g] for g in TARGET_COLS}
            ov, _ = scores_3fold(blend)
            seed_scores.append(ov)
            print(f"seed={seed}: 재최적블렌드 3-fold={ov:.4f}")
        m, sdv = float(np.mean(seed_scores)), float(np.std(seed_scores))
        gap = m - BASE_BLEND_SEEDMEAN
        ratio = gap / sdv if sdv > 0 else float('nan')
        print(f"\nseed 평균={m:.4f}, 표준편차={sdv:.4f} | 3절 기준(0.6316) 대비 {gap:+.4f} ({ratio:+.1f}배), 가중치 {fmt_w(final_w)}")
        print("-> 표준편차의 여러 배(채택 기준)를 넘으면 train/inference 갱신 후 제출로 확인, 아니면 유지.")

최적 조합: {'hidden': (256, 256), 'dropout': 0.35, 'lr': 0.003, 'weight_decay': 0.0001}
baseline : hidden(256, 256) drop0.15 lr0.001 wd0.0001

[seed=42] 재최적블렌드=0.6319 (baseline 0.6323) | fold3=0.6574 (baseline 0.6536)
-> baseline 미달 또는 fold3 역행 -> 채택하지 않음(seed 재검증 생략).


## 6. 종합 해석 (5절)

### 6-1. 결과 — baseline 유지(기각), 새 손실에서도 하이퍼파라미터는 이미 최적

**자동 판정: 채택 안 함.** 순차 탐색으로 나온 최적 조합(hidden 256-256, dropout 0.35, lr 0.003, wd 1e-4)의 재최적 블렌드가 **0.6319로 baseline(0.6323)에 못 미쳐** 5-5에서 seed 재검증 없이 기각됐다.

| 스윕 | fold3 기준 선택 | 관찰 |
|---|---|---|
| 5-2 구조 | (256,256) 유지 | 재최적블렌드 최고는 **(256,256,256)=0.6346(+0.0023)**이지만 fold3이 0.6515로 baseline(0.6536)보다 **-0.0021 역행** |
| 5-3 dropout | 0.35 | fold3만 최고(0.6557), 재최적블렌드로는 0.35가 **최저(0.6306)** — 방향이 갈림 |
| 5-4 lr / wd | lr 0.003 / **wd 1e-4(=baseline)** | wd는 18절 미탐색 후보였는데 baseline값 1e-4가 최적으로 확인됨 |

### 6-2. 핵심 — fold3 안전장치가 외부 phase8 함정을 그대로 걸러냄

가장 유혹적인 신호는 **(256,256,256)의 재최적 블렌드 +0.0023**(22절 개선폭 +0.0021과 비슷한 크기)이었다. 하지만 그 구조는 **fold3(가장 미래)이 baseline보다 -0.0021 역행**했다 — 외부 파이프라인 phase8이 실측으로 겪은 함정("MLP 구조를 키우니 CV +0.0119였지만 홀드아웃 -0.0045로 역행")과 **정확히 같은 패턴**이다. 게다가 이번은 트레이드가 더 나쁘다: **fold3 손실(-0.0021)이 블렌드 CV 이득(+0.0023)과 거의 맞먹어**, 미래(2025 test)에서 순이득이 나온다는 보장이 없고 오히려 역행할 위험이 크다. 18절에서 세운 fold3 병행 확인 원칙 덕분에 "평균/블렌드만 보고 3층 구조를 채택해 미래에 후퇴하는" 선택을 이번에도 미리 걸러냈다.

*(방법론 메모: 5-2~5-4의 `best_*` 자동선택을 fold3 argmax로 뒀는데, dropout 스윕에서 fold3 최고값(0.35)이 재최적블렌드로는 최저라 "블렌드에 나쁜" 값을 집었다. 그래도 최종 5-5 게이트(재최적블렌드 baseline 대비 + fold3 비역행)가 이를 걸러내 안전하게 기각으로 수렴했다.)*

### 6-3. So-what

- **손실을 그룹별로 바꾸고 T_soft를 0.003->0.035로 크게 옮겨도, MLP 구조·dropout·학습률·weight_decay의 최적점은 옮겨지지 않았다** — 18절(옛 pooled 손실)의 "이미 최적" 결론이 새 손실 위에서도 재확인됐다. 확정 모델 v4(hidden 256-256, dropout 0.15, lr 1e-3, wd 1e-4, T_soft 0.035)가 하이퍼파라미터 측면에서 국소 최적에 가깝다.
- **18절에서 미탐색으로 남겨둔 weight_decay도 이번에 해소** — 1e-4(baseline)가 최적, 0/1e-3 모두 그보다 낮았다. MLP 하이퍼파라미터 레버는 여기서 소진.
- **다음 방향**: 하이퍼파라미터가 아니라 **구조적 레버**(그룹별 MLP, 원본 풍속 rolling, FICR 분위수 앙상블 등 HANDOFF 우선순위 2~5)로 넘어갈 시점. 특히 "채점 산식 직접 겨냥" 계열(τ->FICR손실->그룹별손실 3연속 성공)의 연장인 FICR 분위수 앙상블에 승산이 있어 보인다.

## 7. 통합 MLP vs 그룹별 MLP 재검토 — 그룹별 정규화

**질문의 출발**: 5절에서 dropout을 통합 MLP에 하나로 걸면 fold3↑ / 블렌드↓가 상충해 baseline(0.15)이 균형점이었다. 민석님 제안 — **그룹을 따로 학습하면** 각 그룹의 스윗스폿을 따로 잡아 이 상충을 완화할 수 있지 않나?

**근거**: group_3은 라벨이 짧고(2023~) 표본이 적어 과적합 위험이 커 **강한 정규화**가, group_1/2는 표본이 많아 **약한 정규화**가 맞을 수 있다. 통합 MLP는 dropout 하나로 셋을 타협하지만, 그룹별 MLP는 각각 최적을 준다. (블렌드 가중치는 이미 그룹별이지만 MLP 자체의 하이퍼파라미터는 지금 세 그룹 공통이다.)

**주의 — 04번 통합 우세 결정**: 04에서 통합(0.5971) > 그룹별(0.5868)이었지만, 그건 50피처·τ=0.70·FICR손실·actual가중 도입 **전**의 낡은 결정이다. 지금 다시 볼 가치가 있다(외부 파이프라인은 그룹별+179피처로 0.6389). 다만 group_3은 데이터가 적어 단독 학습이 불안정할 수 있으니 seed·fold3로 신중히 검증한다.

**접근(3단계)**: 7-1 동일 하이퍼파라미터로 그룹별 3개 학습 → 통합과 비교(분리 자체의 효과), 7-2 그룹별 dropout 스윕(각 그룹 독립, group_3 강한 정규화 가설 확인), 7-3 그룹별 최적 조합 seed 재검증(통합 v4 seed 평균 0.6316 기준). **22절 그룹별 손실·T_soft=0.035는 그대로 유지.**

**의존성**: 0절 헬퍼만 있으면 단독 실행(필요한 블렌드 함수는 7-1에 재정의). 수정된 `src/nn.py`(그룹별 손실) 전제. **실행 시간**: 7-1(그룹별 3개×3fold=9) + 7-2(3그룹×5dropout×3fold, 그룹별 단독 학습) + 7-3(3seed×3그룹×3fold) ≈ 20~40분. 그리드는 상수로 노출.

### 7-1. 그룹별 MLP 인프라 + 분리 효과 (동일 하이퍼파라미터)

각 그룹 데이터만으로 별도 MLP를 학습하는 `train_group_mlp`를 정의한다(`build_mlp_features`가 group_id 원-핫을 붙이고 `nn.train_mlp`가 그룹별 손실을 그 그룹으로 자동 복원하므로 기존 인프라를 그대로 재사용 — 단일 그룹이면 그 그룹만의 NMAE·FICR로 학습된다). 먼저 baseline 하이퍼파라미터(256-256/0.15/1e-3/wd 1e-4/T_soft 0.035)로 그룹별 3개를 학습해, 통합 MLP(단일 seed 재최적블렌드 0.6323, seed 평균 0.6316)와 **같은 조건에서 분리 자체가 도움되는지** 본다.

In [18]:
# 7절: 통합 MLP vs 그룹별 MLP. 0절만으로 독립 실행되도록 블렌드 함수를 여기서 재정의(1절 것과 동일).
UNIFIED_REOPT_S42 = 0.6323       # 통합 MLP baseline 단일 seed=42 재최적블렌드(1b/5-1)
UNIFIED_BLEND_SEEDMEAN = 0.6316  # 통합 MLP seed 3개 평균(3절) = 최종 판정 기준
GB_T_SOFT, GB_HIDDEN, GB_LR, GB_WD = 0.035, (256, 256), 1e-3, 1e-4
GB_DROPOUT = 0.15
BLEND_GRID_7 = [round(0.1 * i, 1) for i in range(11)]


def fmt_w(w):
    return f"{w['kpx_group_1']:.1f}/{w['kpx_group_2']:.1f}/{w['kpx_group_3']:.1f}"


def scores_3fold_7(pred_by_fold):
    overall = []
    for fold in FOLDS:
        fv = CAT_FOLD[fold["name"]]["fv"]
        s, _, _ = metric(make_answer_df(fv), make_pred_df(fv, pred_by_fold[fold["name"]]))
        overall.append(s)
    return float(np.mean(overall))


def optimize_blend_weights_7(mlp_by_fold, grid=BLEND_GRID_7):
    best_w, best_score = {}, {}
    for g in TARGET_COLS:
        top = (-1.0, None)
        for w in grid:
            fs = []
            for fold in FOLDS:
                fv = CAT_FOLD[fold["name"]]["fv"]
                pred = (1 - w) * CAT_FOLD[fold["name"]]["cat"][g] + w * mlp_by_fold[fold["name"]][g]
                fs.append(single_group_score(fv, g, pred))
            s = float(np.mean(fs))
            if s > top[0]:
                top = (s, w)
        best_score[g], best_w[g] = top
    overall = float(np.mean([best_score[g] for g in TARGET_COLS]))
    return overall, best_w, best_score


def to_long_single(df, g, feature_cols):
    sub = df[df[g].notna()].copy()
    sub["group_id"] = GROUP_ID_MAP[g]
    sub["utilization"] = sub[g] / CAPACITY_KWH[g]
    sub["actual_kwh"] = sub[g]
    return sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols].reset_index(drop=True)


def train_group_mlp(fold_train, g, feature_cols, seed=SEED, T_soft=GB_T_SOFT,
                    hidden=GB_HIDDEN, dropout=GB_DROPOUT, lr=GB_LR, weight_decay=GB_WD, early_stop_frac=0.15):
    long_df = to_long_single(fold_train, g, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]
    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)
    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10
    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10
    model, _, _ = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored, early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, lr=lr, weight_decay=weight_decay,
    )
    return model, mu, sd


def group_mlp_predict(g, seed=SEED, **cfg):
    # 그룹 g만 3-fold 학습 -> {fold_name: 그 그룹 kwh 예측}
    pred_by_fold = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd = train_group_mlp(ft, g, FEATURE_COLS, seed=seed, **cfg)
        pred_by_fold[fold["name"]] = predict_group_mlp(model, fv, g, FEATURE_COLS, mu, sd)
    return pred_by_fold


def group_mlp_cv(seed=SEED, group_cfg=None):
    # 세 그룹 각각 학습 -> {fold: {g: pred}}, 전체 3-fold Score, fold3 단독 Score
    group_cfg = group_cfg or {}
    per_group = {g: group_mlp_predict(g, seed=seed, **group_cfg.get(g, {})) for g in TARGET_COLS}
    mlp_by_fold = {fold["name"]: {g: per_group[g][fold["name"]] for g in TARGET_COLS} for fold in FOLDS}
    overall = scores_3fold_7(mlp_by_fold)
    fv3 = CAT_FOLD["fold3"]["fv"]
    s3, _, _ = metric(make_answer_df(fv3), make_pred_df(fv3, mlp_by_fold["fold3"]))
    return mlp_by_fold, overall, s3


# 분리 효과: 동일 하이퍼파라미터로 그룹별 3개 학습 -> 통합과 비교
gb_mlp, gb_overall, gb_fold3 = group_mlp_cv()
gb_reopt, gb_w, gb_g = optimize_blend_weights_7(gb_mlp)
print(f"[그룹별 MLP, 동일 하이퍼파라미터] MLP전체={gb_overall:.4f} | MLP fold3={gb_fold3:.4f} | 재최적블렌드={gb_reopt:.4f} [w={fmt_w(gb_w)}]")
print(f"[통합 MLP baseline]              단일seed 재최적블렌드={UNIFIED_REOPT_S42:.4f} | seed평균={UNIFIED_BLEND_SEEDMEAN}")
print(f"  차이(그룹별 - 통합, 단일seed) = {gb_reopt - UNIFIED_REOPT_S42:+.4f}")
print("  그룹별 재최적 블렌드 그룹점수: " + ", ".join(f"{g[-1]}={gb_g[g]:.4f}" for g in TARGET_COLS))

[그룹별 MLP, 동일 하이퍼파라미터] MLP전체=0.6173 | MLP fold3=0.6554 | 재최적블렌드=0.6274 [w=0.3/0.6/0.5]
[통합 MLP baseline]              단일seed 재최적블렌드=0.6323 | seed평균=0.6316
  차이(그룹별 - 통합, 단일seed) = -0.0049
  그룹별 재최적 블렌드 그룹점수: 1=0.6456, 2=0.6533, 3=0.5831


### 7-2. g1/g2 전용 튜닝 (g3는 통합) — 블렌드 우선(채택)

7-1에서 그룹별 g1/g2가 통합보다 근소하게 낮았던 건 **동일 baseline 하이퍼파라미터**를 썼기 때문이다. 데이터가 충분한 g1/g2를 **각자 전용으로 dropout→구조를 튜닝**해 통합 수준 이상으로 끌어올릴 수 있는지 본다(g3는 통합 유지). 각 그룹 독립으로 **블렌드 후 그룹점수**(그 그룹이 최종 점수에 기여하는 값)를 최대화하되 fold3(가장 미래)을 병행 확인한다.

**선택 기준(블렌드 우선)**: 블렌드 그룹점수 최고를 고른다. *민석님 제안으로 "블렌드가 동급이면 fold3 좋은 쪽" tiebreak도 별도로 시험했으나(7-3 참고), dropout을 fold3 좇아 올리면 블렌드가 무너져 최종 seed 평균이 오히려 낮았다(0.6334 → 0.6320). "3-fold 블렌드가 주 기준, fold3은 역행 방지용"이라는 원칙이 데이터로 재확인돼 블렌드 우선을 채택한다.*

**목표선**: g1 통합 0.6458 / g2 통합 0.6555 — 전용 튜닝된 그룹별이 이 값 이상이면 하이브리드가 통합 v4를 넘는다.

In [27]:
# 7-2. g1/g2 전용 튜닝 (g3는 통합). 블렌드 그룹점수 우선(채택본) + fold3 병행 출력.
TUNE_GROUPS = ["kpx_group_1", "kpx_group_2"]
GDROP_GRID = [0.05, 0.10, 0.15, 0.25]
GHID_GRID = [(256, 256), (256, 128), (384, 384)]
UNI_GROUP_SCORE = {"kpx_group_1": 0.6458, "kpx_group_2": 0.6555, "kpx_group_3": 0.5957}  # 통합 v4 그룹점수(1b T_soft=0.035)


def group_blend_opt(g, mlp_pred_by_fold, grid=BLEND_GRID_7):
    top = (-1.0, None)
    for w in grid:
        fs = []
        for fold in FOLDS:
            fv = CAT_FOLD[fold["name"]]["fv"]
            pred = (1 - w) * CAT_FOLD[fold["name"]]["cat"][g] + w * mlp_pred_by_fold[fold["name"]]
            fs.append(single_group_score(fv, g, pred))
        s = float(np.mean(fs))
        if s > top[0]:
            top = (s, w)
    fv3 = CAT_FOLD["fold3"]["fv"]
    pred3 = (1 - top[1]) * CAT_FOLD["fold3"]["cat"][g] + top[1] * mlp_pred_by_fold["fold3"]
    f3 = single_group_score(fv3, g, pred3)
    return top[1], top[0], f3


group_best = {}
for g in TUNE_GROUPS:
    print(f"[{g}] 전용 튜닝 (통합 목표선 {UNI_GROUP_SCORE[g]:.4f})")
    best_d, best_ds = GB_DROPOUT, -1.0
    for d in GDROP_GRID:
        preds = group_mlp_predict(g, dropout=d)
        _, bg, bf3 = group_blend_opt(g, preds)
        print(f"  drop={d:.2f} | 블렌드 그룹점수={bg:.4f} | fold3={bf3:.4f}")
        if bg > best_ds:
            best_ds, best_d = bg, d
    print(f"  -> best dropout={best_d}")
    best_h, best_hs, best_hf3 = GB_HIDDEN, -1.0, None
    for h in GHID_GRID:
        preds = group_mlp_predict(g, dropout=best_d, hidden=h)
        _, bg, bf3 = group_blend_opt(g, preds)
        print(f"  hidden={str(h):12} drop={best_d:.2f} | 블렌드 그룹점수={bg:.4f} | fold3={bf3:.4f}")
        if bg > best_hs:
            best_hs, best_h, best_hf3 = bg, h, bf3
    group_best[g] = {"dropout": best_d, "hidden": best_h}
    diff = best_hs - UNI_GROUP_SCORE[g]
    print(f"  -> {g} 최적: dropout={best_d}, hidden={best_h} | 전용 그룹점수={best_hs:.4f}, fold3={best_hf3:.4f} (통합 대비 {diff:+.4f})\n")

print("g1/g2 전용 최적:", group_best)

[kpx_group_1] 전용 튜닝 (통합 목표선 0.6458)
  drop=0.05 | 블렌드 그룹점수=0.6450 | fold3=0.6635
  drop=0.10 | 블렌드 그룹점수=0.6463 | fold3=0.6652
  drop=0.15 | 블렌드 그룹점수=0.6456 | fold3=0.6647
  drop=0.25 | 블렌드 그룹점수=0.6453 | fold3=0.6655
  -> best dropout=0.1
  hidden=(256, 256)   drop=0.10 | 블렌드 그룹점수=0.6463 | fold3=0.6652
  hidden=(256, 128)   drop=0.10 | 블렌드 그룹점수=0.6451 | fold3=0.6656
  hidden=(384, 384)   drop=0.10 | 블렌드 그룹점수=0.6451 | fold3=0.6662
  -> kpx_group_1 최적: dropout=0.1, hidden=(256, 256) | 전용 그룹점수=0.6463, fold3=0.6652 (통합 대비 +0.0005)

[kpx_group_2] 전용 튜닝 (통합 목표선 0.6555)
  drop=0.05 | 블렌드 그룹점수=0.6534 | fold3=0.6779
  drop=0.10 | 블렌드 그룹점수=0.6534 | fold3=0.6787
  drop=0.15 | 블렌드 그룹점수=0.6533 | fold3=0.6807
  drop=0.25 | 블렌드 그룹점수=0.6532 | fold3=0.6831
  -> best dropout=0.05
  hidden=(256, 256)   drop=0.05 | 블렌드 그룹점수=0.6534 | fold3=0.6779
  hidden=(256, 128)   drop=0.05 | 블렌드 그룹점수=0.6527 | fold3=0.6810
  hidden=(384, 384)   drop=0.05 | 블렌드 그룹점수=0.6571 | fold3=0.6843
  -> kpx_group_2 최적: dropout=0.05

### 7-3. 최종 하이브리드 seed 재검증

7-2에서 찾은 g1/g2 전용 하이퍼파라미터로 그룹별 MLP를, g3는 통합 MLP를 학습해 조립한 하이브리드를 seed 3개(42/7/2024)로 재검증한다. 통합 v4 seed 평균(0.6316)과 표준편차 대비 개선폭을 보고, 표준편차의 여러 배를 넘으면 채택(train/inference를 이 하이브리드 구조로)을 검토한다.

In [28]:
# 7-3. 최종 하이브리드(g1/g2 전용 + g3 통합) seed 재검증
PREV_BLEND_FIRST = 0.6334  # 앞선 '블렌드 우선' 버전의 하이브리드 seed 평균(+0.0018/6.8배) — fold3 tiebreak 버전과 비교용
seed_scores = []
for seed in [42, 7, 2024]:
    g12 = {g: group_mlp_predict(g, seed=seed, **group_best[g]) for g in TUNE_GROUPS}
    uni_g3 = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=seed, T_soft=GB_T_SOFT)
        uni_g3[fold["name"]] = predict_group_mlp(model, fv, "kpx_group_3", FEATURE_COLS, mu, sd)
    hyb = {}
    for fold in FOLDS:
        fn = fold["name"]
        hyb[fn] = {"kpx_group_1": g12["kpx_group_1"][fn], "kpx_group_2": g12["kpx_group_2"][fn], "kpx_group_3": uni_g3[fn]}
    reopt, w, g_scores = optimize_blend_weights_7(hyb)
    seed_scores.append(reopt)
    print(f"seed={seed}: 하이브리드 재최적블렌드={reopt:.4f} [w={fmt_w(w)}] "
          f"(g1={g_scores['kpx_group_1']:.4f}, g2={g_scores['kpx_group_2']:.4f}, g3={g_scores['kpx_group_3']:.4f})")

m, sdv = float(np.mean(seed_scores)), float(np.std(seed_scores))
gap = m - UNIFIED_BLEND_SEEDMEAN
ratio = gap / sdv if sdv > 0 else float('nan')
print(f"\n[fold3 tiebreak] 하이브리드 seed 평균={m:.4f}, 표준편차={sdv:.4f} | 통합 v4(0.6316) 대비 {gap:+.4f} ({ratio:+.1f}배)")
print(f"[블렌드 우선 버전(이전)]         seed 평균={PREV_BLEND_FIRST} (+0.0018/6.8배)")
print(f"-> 둘 중 높은 쪽을 채택. 통합 v4를 표준편차 여러 배로 넘으면 train/inference를 g1/g2 그룹별+g3 통합 구조로.")

seed=42: 하이브리드 재최적블렌드=0.6330 [w=0.3/0.7/0.8] (g1=0.6463, g2=0.6571, g3=0.5957)
seed=7: 하이브리드 재최적블렌드=0.6337 [w=0.4/0.6/0.9] (g1=0.6489, g2=0.6534, g3=0.5987)
seed=2024: 하이브리드 재최적블렌드=0.6334 [w=0.3/0.7/1.0] (g1=0.6476, g2=0.6550, g3=0.5978)

[fold3 tiebreak] 하이브리드 seed 평균=0.6334, 표준편차=0.0003 | 통합 v4(0.6316) 대비 +0.0018 (+6.8배)
[블렌드 우선 버전(이전)]         seed 평균=0.6334 (+0.0018/6.8배)
-> 둘 중 높은 쪽을 채택. 통합 v4를 표준편차 여러 배로 넘으면 train/inference를 g1/g2 그룹별+g3 통합 구조로.


### 7-4. 최종 블렌드 가중치 확정 (seed 앙상블 예측 위)

train/inference는 CatBoost·MLP 모두 seed 3개(42/7/2024) 앙상블을 쓰므로, 블렌드 가중치도 **seed 앙상블 예측 위에서** 한 번 확정해야 정합한다(7-3의 seed별 재최적 w는 조금씩 달랐다). g1/g2 전용 그룹별 MLP·g3 통합 MLP를 각각 seed 3개 앙상블하고, CatBoost도 seed 3개 앙상블한 뒤 그룹별 블렌드 가중치를 재최적한다. 이 `FINAL_BLEND_W`를 train.ipynb의 `BLEND_WEIGHTS`에 하드코딩한다.

In [29]:
# 7-4. 최종 블렌드 가중치 확정 — seed 3개 앙상블(CatBoost·MLP 모두) 예측 위에서 그룹별 블렌드 재최적
SEEDS_FINAL = [42, 7, 2024]

# (1) MLP seed 앙상블: g1/g2 전용 그룹별 + g3 통합
mlp_ens = {fold["name"]: {} for fold in FOLDS}
for g in TUNE_GROUPS:  # g1, g2 전용
    seed_preds = [group_mlp_predict(g, seed=s, **group_best[g]) for s in SEEDS_FINAL]
    for fold in FOLDS:
        mlp_ens[fold["name"]][g] = np.mean([sp[fold["name"]] for sp in seed_preds], axis=0)
uni_seed = []
for s in SEEDS_FINAL:  # g3 통합
    d = {}
    for fold in FOLDS:
        ft, _ = make_fold_frames(fold)
        fv = CAT_FOLD[fold["name"]]["fv"]
        model, mu, sd, _ = train_mlp_fold(ft, FEATURE_COLS, seed=s, T_soft=GB_T_SOFT)
        d[fold["name"]] = predict_group_mlp(model, fv, "kpx_group_3", FEATURE_COLS, mu, sd)
    uni_seed.append(d)
for fold in FOLDS:
    mlp_ens[fold["name"]]["kpx_group_3"] = np.mean([sp[fold["name"]] for sp in uni_seed], axis=0)

# (2) CatBoost seed 앙상블(블렌드 정합용 — CAT_FOLD는 seed42 단일이라 새로 앙상블)
cat_ens = {fold["name"]: {} for fold in FOLDS}
for fold in FOLDS:
    ft, _ = make_fold_frames(fold)
    fv = CAT_FOLD[fold["name"]]["fv"]
    seed_models = [train_fold_model(ft, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True, seed=s) for s in SEEDS_FINAL]
    for g in TARGET_COLS:
        cat_ens[fold["name"]][g] = np.mean([predict_group(m, fv, g) for m in seed_models], axis=0)

# (3) 그룹별 블렌드 가중치 재최적 (seed 앙상블 예측 위)
FINAL_BLEND_W = {}
final_group_scores = {}
for g in TARGET_COLS:
    top = (-1.0, None)
    for w in BLEND_GRID_7:
        fs = []
        for fold in FOLDS:
            fv = CAT_FOLD[fold["name"]]["fv"]
            pred = (1 - w) * cat_ens[fold["name"]][g] + w * mlp_ens[fold["name"]][g]
            fs.append(single_group_score(fv, g, pred))
        s = float(np.mean(fs))
        if s > top[0]:
            top = (s, w)
    final_group_scores[g], FINAL_BLEND_W[g] = top[0], top[1]

overall = float(np.mean([final_group_scores[g] for g in TARGET_COLS]))
print("FINAL_BLEND_W (train.ipynb BLEND_WEIGHTS에 하드코딩):")
for g in TARGET_COLS:
    print(f"  {g}: w={FINAL_BLEND_W[g]:.1f} | 그룹점수={final_group_scores[g]:.4f}")
print(f"seed 앙상블 하이브리드 3-fold 전체 = {overall:.4f} (통합 v4 seed 평균 0.6316 대비 {overall-0.6316:+.4f})")

FINAL_BLEND_W (train.ipynb BLEND_WEIGHTS에 하드코딩):
  kpx_group_1: w=0.3 | 그룹점수=0.6461
  kpx_group_2: w=0.6 | 그룹점수=0.6561
  kpx_group_3: w=1.0 | 그룹점수=0.6003
seed 앙상블 하이브리드 3-fold 전체 = 0.6342 (통합 v4 seed 평균 0.6316 대비 +0.0026)


## 8. 종합 해석 (7절)

### 8-1. 결과 — 하이브리드(g1/g2 전용 그룹별 + g3 통합)가 통합 v4를 이김

| 단계 | 결과 |
|---|---|
| 7-1 전면 그룹별(동일 hp) | 재최적블렌드 0.6274 = 통합 −0.0049. **g3만 −0.0126**(표본 부족), g1/g2는 거의 동일 |
| 7-2 g1/g2 전용 튜닝 | g1 drop0.10·(256,256) → +0.0005 / **g2 drop0.05·(384,384) → +0.0016** (통합 대비) |
| **7-3 하이브리드 seed 재검증** | **seed 평균 0.6334(표준편차 0.0003), 통합 v4(0.6316) 대비 +0.0018 / 6.8배** ✅ |

**채택 기준(표준편차 6배)을 넘겼다** — 9절(47.6배)·14절(6.2배)에 이은 세 번째 유의미 개선. 구조: **group_1/2는 각자 전용 튜닝한 그룹별 MLP, group_3은 통합 MLP**, 그 위에 CatBoost 그룹별 블렌드.

### 8-2. 왜 이 하이브리드가 통하나 — "정답은 그룹마다 다르다"

- **group_3(통합)**: 라벨이 짧아(2023~) 표본이 적다. 단독 학습하면 −0.0126로 무너지지만(7-1), 통합에선 group_1/2의 풍부한 데이터에서 패턴을 빌려 배운다. → 통합 유지가 정답.
- **group_1/2(전용 그룹별)**: 데이터가 충분해 분리 손해가 거의 없고(−0.0002/−0.0022), **전용 튜닝으로 통합을 넘어선다**(특히 g2는 (384,384) 구조가 블렌드·fold3를 동시에 올림). → 그룹별+전용 튜닝이 정답.
- 04번의 이분법(전부 통합 vs 전부 그룹별)이 놓친 지점: **데이터가 많은 그룹은 그룹별, 적은 그룹은 통합**이라는 그룹별 최적 구조. 04 결정(통합 우세)은 50피처·τ·FICR손실 전이라 낡았고, 실제로는 혼합이 최선이었다.

### 8-3. fold3 tiebreak 검증 (민석님 지적) — 블렌드 우선이 옳았음

7-2 선택에서 "블렌드가 동급이면 fold3 좋은 쪽" tiebreak를 별도로 시험했다. 결과: dropout을 fold3 좇아 0.25로 올리니 **블렌드(3-fold)가 무너져** g2가 0.6571→0.6532(통합보다도 −0.0023), 하이브리드 seed 평균이 **0.6334→0.6320(+0.0018→+0.0004)로 오히려 하락**했다. fold3은 단일 fold(2024 하반기)라 노이즈가 커, 그것만 최대화하면 전체 3-fold를 희생한다. **"3-fold 블렌드가 주 기준, fold3은 역행 방지용"이라는 이 프로젝트 원칙이 데이터로 재확인**됐고, 블렌드 우선(0.6334)을 채택했다.

### 8-4. So-what & 다음

- **하이브리드가 통합 v4(LB 0.629122)를 CV +0.0018/6.8배로 이김** → 새 최고 후보(v5). 6.8배는 v4의 1.7배보다 훨씬 강한 신호라 LB에서 먹힐 개연성이 높다. 단 **train/inference를 구조 변경**(g1/g2 전용 그룹별 MLP + g3 통합 MLP)해야 하는 중간 규모 작업.
- 채택 시: g1(drop0.10, 256-256), g2(drop0.05, 384-384) 전용 그룹별 MLP + g3 통합 MLP(baseline hp), 모두 T_soft=0.035 그룹별 손실·seed 3개 앙상블, CatBoost와 그룹별 블렌드 재최적.

## 9. 원본 풍속의 발표분 내 rolling·diff 피처 (HANDOFF 우선순위 2)

**가설**: 외부 파이프라인(Public 0.63886)의 feature importance 1위가 `pc_pred`의 5시간 이동평균(roll5)이었다. 17절에서 이 착안을 `pc_pred`에만 적용했으나 baseline을 못 넘었다(-0.0004). 그러나 **원본 풍속(`gfs_ws850hpa`/`gfs_ws100m`/`ldaps_ws10m`)의 rolling·diff는 한 번도 시도하지 않았다** — permutation importance 1위가 `gfs_ws850hpa`인데 이 컴럼엔 diff조차 없다.

**물리적 근거**:
- **발전량 ∝ 풍속³** — 풍속의 시간 평활화(rolling)는 NWP 타이밍 오차(전선 통과 시점 어긍남)를 완화한다. 순간 예보값보다 몇 시간 평균이 실제 발전 구간을 더 안정적으로 대표.
- **diff(발표분 내 시간변화율)** — 전선 통과 등 급변 구간 포착(`wind-domain-features` 5절). `gfs_ws100m`·`ldaps_ws10m`엔 이미 `_diff_prev`가 있어 효과가 검증됐는데, **산 정상 자유대기 풍황을 대표하는 `gfs_ws850hpa`엔 없다.**

**누수 소명**: rolling·diff 모두 **같은 발표분(`data_available_kst_dtm`) 경계 안에서만** 계산한다(03_features `add_issuance_diff`와 동일 groupby). 한 발표분의 24시간은 D-1 13:00에 동시 공개되므로 발표분 내 다른 시각 참조는 누수가 아니다(leakage-guard "예보값의 발표분 내 diff/rolling ✅"). trailing(과거방향)이라 실제로는 필요 이상 보수적이다. train/test 동일 로직으로 재현 가능.

> **실행·결론(2026-07-25)**: 이 가설의 실제 실험은 "피처 확정 후 CatBoost·MLP 전체 재튜닝"으로 별도 노트북 `05_tuning_4.ipynb`에서 수행했다(민석님 판단: 피처 바꾸면 결합된 knob도 다 재튜닝해야 공정). **결과는 기각** — 아래 9-1 결과 참조. `05_tuning_4`는 폐기했고(폐기 가능 설계), 핵심 수치만 여기에 남긴다.

## 9-1. 결과 및 종합 해석 (실행: 05_tuning_4, 기각)

**실험 설계**: 증강 피처(61개 = 50 + roll 10 + diff 1)를 확정 입력으로 고정하고 CatBoost(Optuna 풀 재탐색: τ+depth/l2/rsm/subsample/min_data_in_leaf/lr)와 통합 MLP(구조→dropout→lr→wd→T_soft 순차 스윕, fold3 argmax)를 처음부터 재튜닝. baseline(50피처, 현재 확정설정)도 같은 실행에서 재계산해 환경차 상쇄. seed 3개(42/7/2024) 재검증.

**측정 신뢰**: baseline(50, 기준 MLP) seed 평균 = **0.6316** — 과거 3절 기록(0.6316)과 정확히 재현.

**1차 관찰(유혹적이었던 헤드라인)**: 증강+재튜닝 seed 평균 **0.6352**, baseline 0.6316 대비 **+0.0036(표준편차 2.2배)**, seed 3개 완전 분리. CatBoost Optuna도 이번엔 최적점이 움직였다(depth 10·lr 0.0147·τ 0.738 → CatBoost 단독 +0.0040). 5절에서 fold3 역행으로 기각됐던 3층 MLP가 이번엔 채택됨((256,256,256)/dropout 0.25/lr 0.002/wd 0).

**디컨파운드(핵심)**: 재튜닝 MLP 설정을 **피처 없이 baseline 50개에 그대로 적용**해 피처 순효과를 격리:

| 구성 | seed평균 | fold3(blend) |
|---|---:|---:|
| baseline(50) + 기준 MLP | 0.6316 | 0.6573 |
| baseline(50) + **재튜닝 MLP** | 0.6340 (**+0.0024**) | 0.6568 (**−0.0005**) |
| 증강(61) + 재튜닝 MLP | 0.6352 (features **+0.0012/1.0배**) | 0.6586 (+0.0018) |

- **피처 순효과 = +0.0012(표준편차 1.0배) = 노이즈.** 헤드라인 +0.0036의 대부분(+0.0024)은 "5절이 놓친 3층 MLP 설정" 덕이지 피처 덕이 아니었다. **피처 실험 7번째 실패**(11·12·13·16·17-raw·17-roll에 이어). 17절 결론("이미 성숙한 CatBoost가 원시 풍속으로 파워커브류 관계를 이미 근사")과 동일 패턴.
- CatBoost Optuna 단독 +0.0040도 블렌드에선 +0.0010으로 희석(블렌드가 MLP 비중 높음: g3=1.0 전량 MLP) → 코어 튜닝 이득이 최종 지표에 거의 안 옮겨짐(8절 결론 재확인).

**부수 관찰(별도 기록, 미채택)**: 재튜닝 3층 MLP가 **baseline 50개에서 seed평균을 +0.0024(std 0.0005, ~2.8배)** 올렸다 — 그러나 **fold3는 −0.0005로 flat/역행**. 5절이 3층을 fold3 역행으로 기각했던 것과 같은 "CV↑·미래 flat" 지문. fold3가 뒷받침하지 않으므로 채택하지 않음(이 프로젝트가 반복해 데인 CV 과적합형). 만약 이 리드를 다시 본다면 fold3의 seed 분산까지 재고 하이브리드 구조에서 검증할 것.

**결정(민석님)**: 피처 기각, `05_tuning_4.ipynb` 폐기(삭제), v5(하이브리드) 유지. train/inference·확정 모델 무변경. 이로써 "손실·구조·피처" 세 레버 중 피처 풍부화(우선순위 2)도 마감 — 남은 후보는 우선순위 5(FICR 분위수 앙상블, 채점 산식 직접 겨냥 계열).